# Snuffelfiets kwartaalrapportages

## Settings.

In [ ]:
# Imports and convenience functions.

from pathlib import Path

import numpy as np
import pandas as pd

from src.snuffelfiets import inlezen, opschonen, analyse, plotting


def get_period_info(period_spec, year, quarter, month):
    """Return an id-string and the months for the chosen period."""

    quarters = {
        'Q1': [1, 2, 3], 
        'Q2': [4, 5, 6], 
        'Q3': [7, 8, 9], 
        'Q4': [10, 11, 12], 
    }
    periods = {
        'jaar': {
            'period_id': f'{year}',
            'months': list(range(1, 13)),
        },
        'kwartaal': {
            'period_id': f'{year}_{quarter}',
            'months': quarters[quarter],
        },
        'maand': {
            'period_id': f'{year}_{month:02d}',
            'months': [month],
        }
    }
    period_id = periods[period_spec]['period_id']
    months = periods[period_spec]['months']

    return period_id, months


def get_center():
    """Return coordinates for the center of provincie Utrecht."""

    # coordinaten van de uiterste punten van de provincie Utrecht
    b = {
        'N': [52.303634, 5.013507],
        'Z':  [51.858631, 5.040462],
        'O':  [51.954780, 5.627990],
        'W':  [52.226808, 4.794457],
    }

    # het berekende middelpunt van de provincie Utrecht
    center = {
        'lat': b['Z'][0] + 0.5 * (b['N'][0] - b['Z'][0]),
        'lon': b['W'][1] + 0.5 * (b['O'][1] - b['W'][1]),
    }

    return center


def get_layers(data_directory):
    """Import Utrecht province and township polygons"""

    # Download the polygons.
    filepaths = plotting.download_borders_utrecht(data_directory)

    # Extract the relevant polygons.
    provincies, gemeenten = plotting.get_borders_utrecht(data_directory, *filepaths)

    # Enter into dictionary in the mapbox_layers format.
    mapbox_layers = [{
        "name": "Gemeenten",
        "below": 'traces',
        "sourcetype": "geojson",
        "type": "line",
        "color": "gray",
        "source": gemeenten,
        }, 
        {
        "name": "Provincies",
        "below": 'traces',
        "sourcetype": "geojson",
        "type": "line",
        "color": "red",
        "source": provincies,
        }]

    return mapbox_layers


In [ ]:
# API settings.
api_key = ''                    # de API key

# File system settings.
prefix = 'api_gegevens'         # de prefix voor de csv-datafiles; default format <prefix>_<yyyy>_<mm>.csv
data_directory = Path(
    Path('~').expanduser(),
    'Documents',
    'MCUdataclub',
    'Snuffelfiets_data',
    'rapportage',
    )                           # het pad naar de folder met de data; dit wordt ook de parent folder van de output
Path.mkdir(data_directory, parents=True, exist_ok=True)

# Date range settings.
period_spec = 'kwartaal'        # de periode waar het rapport over gaat: 'maand', 'kwartaal' of 'jaar'?
year = 2024                     # het jaar waar het rapport over gaat
quarter = 'Q2'                  # het kwartaal waar het rapport over gaat: 'Q1', 'Q2', 'Q3', 'Q4'
month = 6                       # de maand waar het rapport over gaat: 1, 2, 3, ..., 10, 11, 12


# Data processing settings.
error_code_selection = []       # te verwijderen error codes; [] => behoud alleen error_code=0
rit_splitter_interval = 1800    # het interval tussen ritten (s)
ritfilters = dict(
    min_measurements=2,             # #
    max_duration=360,               # minutes
    max_distance=200,               # kilometers
    min_average_speed=1,            # km/h
    max_average_speed=35,           # km/h
    )                           # criteria waaraan ritten moeten voldoen
threshold_pm2_5 = 100           # de cutoff value voor geldige PM2.5 waardes


# Hexbin plot settings.
mapbox_center = get_center()    # het middelpunt van de provincie Utrecht
mapbox_extent = 1               # de breedte rondom de mapbox_center [deg lat/lon]; datapunten buiten deze breedte/lengtegraden worden verwijderd
hexagon_size = 0.010            # de grootte van de hexagons in de hexbin plot
hexbin_args = {
    'agg_func': np.nanmean,
    'color_continuous_scale': plotting.discrete_colorscale(),
    'range_color': [0, threshold_pm2_5],
    'min_count': 2,
    'animation_frame': None,
    'width': 1920,
    'height': 1080,
    'opacity': 1.0,
    'zoom': 10,
    'center': mapbox_center,
    }                           # parameters for creating the hexbin plot
mapbox_layers = get_layers(data_directory)
layout_args = {
    'mapbox_style': 'open-street-map',
    'coloraxis_showscale': False,
    'mapbox_layers': mapbox_layers,
    }                           # parameters for the layout of the hexbin plot

# Directories.
period_id, months = get_period_info(period_spec, year, quarter, month)
output_directory = Path(data_directory, period_id)
output_directory.mkdir(parents=True, exist_ok=True)

print(f'Analysing {period_spec} {period_id}; writing output to {output_directory}')


## Read data from monthly CSV's.


In [ ]:
# Read the data for the selected period (in monthly chunks).

dfs = []

for m in months:

    filename = f'{prefix}_{year}-{m:02d}.csv'
    p = Path(data_directory, filename)

    try:

        # try to read the data from saved csv's
        df = pd.read_csv(p)

    except FileNotFoundError:

        # download the data if the file does not exist
        inlezen.monthly_csv_dump(api_key, year, m, data_directory, prefix)

        df = pd.read_csv(p)

    dfs.append(df)

df = pd.concat(dfs)

print(f'Read {df.shape[0]} measurements.')


## Data preparation

In [ ]:
# Get some insight in the error modes present in the dataset.
opschonen.analyse_errors(df)


In [ ]:
# Drop the errors.
df = opschonen.verwijder_errors(df, error_codes=error_code_selection)


In [ ]:
# Convert timestamps to datetime objects and add dt columns.
df = analyse.bewerk_timestamp(df, split=True)


In [ ]:
# Split measurements into rides and add cycle stat columns.
df = analyse.split_in_ritten(df, t_seconden=rit_splitter_interval)


In [ ]:
# Filter the rides.
df = analyse.filter_ritten(df, **ritfilters)


## Summary Snuffelfiets statistics.

In [ ]:
def printfun(period, sumstats):

    print(f'\n==== {period} ====\n')

    print(f"Aantal fietsers: {sumstats['fietsers']['N']}\n")

    print(f"{' ':20} {'totaal':12} {'gemiddeld':12} {'topper':12}")
    print(f'-' * 56)
    stat = 'uren'
    print(f"FIETSTIJD [uur]:  {sumstats[stat]['N']:12f} {sumstats[stat]['G']:12f} {sumstats[stat]['M']:12f}")
    stat = 'afstand'
    print(f"AFSTAND    [km]:  {sumstats[stat]['N']:12f} {sumstats[stat]['G']:12f} {sumstats[stat]['M']:12f}")
    stat = 'ritten'
    print(f"RITTEN      [#]:  {sumstats[stat]['N']:12f} {sumstats[stat]['G']:12f} {sumstats[stat]['M']:12f}")


In [ ]:
# Print the summary statistics for the period.
sumstats = analyse.summary_stats(df)
printfun(period_id, sumstats)


In [ ]:
# Print the summary statistics for the months in the quarter.
for m, dfm in df.groupby('month'):
    sumstats = analyse.summary_stats(dfm)
    printfun(f'{year}{m:02d}', sumstats)


## Air quality evaluation

In [ ]:
# Plot categorical hist of PM2.5 values.
def plot_hbar_cat(df, bins, labels, title=''):
    df[f'{var}_cat'] = pd.cut(df[var], bins=bins, labels=labels)
    ax = df[['entity_id', f'{var}_cat']].groupby(f'{var}_cat', observed=False).count().plot.barh(stacked=True, legend=False)
    ax.invert_yaxis()
    ax.axes.get_yaxis().get_label().set_visible(False)
    ax.axes.get_xaxis().set_label_text("Measurement count")
    ax.set_title(title)

var = 'pm2_5'
bins = [0, 1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
labels = [0, 1, 2, 5, 10, 20, 50, 100, 200, 500]
plot_hbar_cat(df, bins, labels, title='PM2.5')


In [ ]:
# Limit the PM2.5 values. FIXME: we need a better and validated method
df['pm2_5'][df['pm2_5'] >= threshold_pm2_5] = np.nan

# Plot the histogram.
df['pm2_5'].hist(bins=200, grid=False)


In [ ]:
## Hexbin plots of PM2.5

# Remove datapoints outside of the map area 
#   because it would take a very long time to process large areas.
#   TODO: doe dit op ritniveau (verwijderen ritten deels of geheel buiten de target area)
latlon = {
    'latitude': {'center': mapbox_center['lat'], 'extent': mapbox_extent},
    'longitude': {'center': mapbox_center['lon'], 'extent': mapbox_extent},
}
df = opschonen.filter_lat_lon(df, latlon)

# Plot the data for the period.
hexbin_args['title'] = period_id
fig = plotting.hexbin_mapbox(df, hexagon_size, hexbin_args, layout_args)

# Save image
filestem = f'{period_id}_hexbin'
output_stem = Path(output_directory, filestem)
fig.write_html(f"{output_stem}.html")
fig.write_image(f"{output_stem}.png")
fig.write_image(f"{output_stem}.pdf")

# Plot the data for each month.
for m, dfm in df.groupby('month'):

    yyyymm = f'{year}{m:02d}'
    hexbin_args['title'] = yyyymm
    fig = plotting.hexbin_mapbox(dfm, hexagon_size, hexbin_args, layout_args)

    # Save image
    filestem = f'{yyyymm}_hexbin'
    output_stem = Path(output_directory, filestem)
    fig.write_html(f"{output_stem}.html")
    fig.write_image(f"{output_stem}.png")
    fig.write_image(f"{output_stem}.pdf")
